# Training — protease cleavage prediction

Input: `windows_<model>.npy` + `training_pairs_aligned.csv` from notebook 01.

This notebook does five things, in the order that matters:

1. Cluster substrates by sequence identity, split by cluster (**not** at random)
2. Train LightGBM, report AUPRC
3. Run three ablations that catch the model cheating
4. Leave-one-enzyme-out — the honest generalization number
5. Calibrate, so outputs are real probabilities

**Read section 4 before you believe any number this produces.**

CPU is fine. Whole notebook runs in ~15 minutes.

## 1. Setup

In [ ]:
!pip -q install lightgbm scikit-learn pandas numpy

In [ ]:
import numpy as np, pandas as pd, json, os, subprocess, warnings
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.isotonic import IsotonicRegression
warnings.filterwarnings("ignore")
RNG = 0
np.random.seed(RNG)

## 2. Load the cache

Upload `windows_*.npy`, `training_pairs_aligned.csv`, `cache_meta.json`
and `combined_clean.csv` (needed for the sequences in step 3).

If they're on Drive, mount instead of uploading — the npy is several GB.

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SRC = "/content/drive/MyDrive/protease_cache"
else:
    from google.colab import files
    files.upload()
    SRC = "."

meta    = json.load(open(f"{SRC}/cache_meta.json"))
pairs   = pd.read_csv(f"{SRC}/training_pairs_aligned.csv")
tag     = meta["model"].replace("esm2_","").replace("_UR50D","")
windows = np.load(f"{SRC}/windows_{tag}.npy", mmap_mode="r")  # lazy: stays on disk
clean   = pd.read_csv("combined_clean.csv") if os.path.exists("combined_clean.csv") else pd.read_csv(f"{SRC}/combined_clean.csv")

assert len(pairs) == len(windows) == meta["rows"], "cache and CSV are out of sync"
print(f"{len(pairs):,} rows | {int(pairs.label.sum()):,} positives "
      f"({pairs.label.mean():.3%}) | {pairs.merops_id.nunique()} enzymes | dim {meta['dim']}")
BASELINE = pairs.label.mean()
print(f"\nRandom-guess AUPRC = {BASELINE:.4f}  <- every result below must beat this")

## 3. Cluster-based splits

Your substrates include near-duplicates — two actin isoforms differing by a few
residues, with identical cleavage sites. If one lands in train and the other in
test, the model recognises it and your score is fiction.

So: cluster substrates at 30% sequence identity, then assign **whole clusters**
to train or test. MMseqs2 does this properly; if it won't install, the fallback
uses 5-mer Jaccard similarity, which is cruder but still far better than random.

In [ ]:
seqs = clean.drop_duplicates("substrate_uniprot").set_index("substrate_uniprot").substrate_sequence.to_dict()
uids = sorted(set(pairs.substrate_uniprot))
print(f"{len(uids):,} substrates to cluster")

with open("subs.fasta","w") as f:
    for u in uids:
        f.write(f">{u}\n{seqs[u]}\n")

def mmseqs_cluster():
    subprocess.run("wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz && tar xzf mmseqs-linux-avx2.tar.gz",
                   shell=True, check=True, timeout=600)
    subprocess.run("./mmseqs/bin/mmseqs easy-cluster subs.fasta clu tmp "
                   "--min-seq-id 0.3 -c 0.5 --cov-mode 1 -v 0",
                   shell=True, check=True, timeout=1800)
    m = {}
    for line in open("clu_cluster.tsv"):
        rep, mem = line.split()
        m[mem] = rep
    return m

def kmer_cluster(k=5, thresh=0.3):
    from collections import defaultdict
    sets = {u: set(seqs[u][i:i+k] for i in range(len(seqs[u])-k+1)) for u in uids}
    inv = defaultdict(list)
    for u, s in sets.items():
        for km in list(s)[:400]:
            inv[km].append(u)
    parent = {u: u for u in uids}
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for u in uids:
        cand = set()
        for km in list(sets[u])[:400]:
            cand.update(inv[km])
        for v in cand:
            if v <= u: continue
            inter = len(sets[u] & sets[v])
            if inter / min(len(sets[u]), len(sets[v]) or 1) >= thresh:
                parent[find(u)] = find(v)
    return {u: find(u) for u in uids}

try:
    clusters = mmseqs_cluster(); method = "MMseqs2 (30% identity)"
except Exception as e:
    print(f"MMseqs2 unavailable ({type(e).__name__}); using k-mer fallback")
    clusters = kmer_cluster(); method = "5-mer Jaccard fallback"

pairs["cluster"] = pairs.substrate_uniprot.map(clusters)
print(f"{method}: {pairs.cluster.nunique():,} clusters from {len(uids):,} substrates")
top = pairs.cluster.value_counts().head(3)
print(f"largest clusters hold {top.values} rows")

In [ ]:
from sklearn.model_selection import GroupKFold

def cluster_split(df, n_splits=5, fold=0):
    gkf = GroupKFold(n_splits=n_splits)
    tr, te = list(gkf.split(df, df.label, groups=df.cluster))[fold]
    return tr, te

tr_idx, te_idx = cluster_split(pairs)
print(f"train {len(tr_idx):,} ({pairs.label.iloc[tr_idx].mean():.3%} pos) | "
      f"test {len(te_idx):,} ({pairs.label.iloc[te_idx].mean():.3%} pos)")
assert not (set(pairs.cluster.iloc[tr_idx]) & set(pairs.cluster.iloc[te_idx])), "cluster leak!"
print("no cluster overlap between train and test")

## 4. Build features

Flatten the 8x1280 window into one row, append a one-hot enzyme code.

One-hot is deliberate: 29 enzymes means the model can only ever memorise 29
things. It's the honest floor. If a real ESM enzyme embedding can't beat 29 bits
later, that embedding isn't contributing anything.

PCA is optional. 10,240 features on 194k rows is wide for trees; reducing each
position to 128 dims usually costs almost nothing and trains 5x faster. Try it
if the full version is slow.

In [ ]:
# ---- memory-safe feature build ----
# Full float32 features would be 7.95 GB and hstack doubles it -> Colab OOM.
# So: fit IncrementalPCA on a residue subsample, then transform in chunks
# directly into one preallocated array. Peak RAM stays ~2 GB.

PCA_DIM  = 128        # 64 if still tight; 256 if you have a high-RAM runtime
CHUNK    = 20_000     # rows per chunk; lower if memory is tight

from sklearn.decomposition import IncrementalPCA

D = meta["dim"]; Wn = meta["window"]
PCA_DIM = min(PCA_DIM, D)          # guard: cannot exceed embedding dim
N = len(pairs)
enz_codes = sorted(pairs.merops_id.unique())
enz_idx   = {e: i for i, e in enumerate(enz_codes)}
E         = len(enz_codes)

# 1. fit PCA on a random sample of residue vectors (never materialise all of them)
rs   = np.random.RandomState(RNG)
rows = rs.choice(N, size=min(30_000, N), replace=False)
sub  = windows[np.sort(rows)].reshape(-1, D).astype(np.float32)
ipca = IncrementalPCA(n_components=PCA_DIM,
                      batch_size=max(4096, PCA_DIM * 2)).fit(sub)
ev   = ipca.explained_variance_ratio_.sum()
del sub
print(f"PCA {D} -> {PCA_DIM}, explained variance {ev:.1%}")

# 2. preallocate the final matrix once, fill it in place
n_feat = Wn * PCA_DIM + E
X = np.zeros((N, n_feat), dtype=np.float32)
print(f"allocating X {X.shape} = {X.nbytes/1e9:.2f} GB")

for s in range(0, N, CHUNK):
    e = min(s + CHUNK, N)
    blk = windows[s:e].reshape(-1, D).astype(np.float32)
    X[s:e, :Wn*PCA_DIM] = ipca.transform(blk).reshape(e-s, Wn*PCA_DIM)
    del blk

X[np.arange(N), Wn*PCA_DIM + pairs.merops_id.map(enz_idx).values] = 1.0
y = pairs.label.values

WIN_FEATS = Wn * PCA_DIM          # used by the ablation cells
print(f"X {X.shape} ({X.nbytes/1e9:.2f} GB) | window feats {WIN_FEATS}, enzyme feats {E}")

import gc; gc.collect()

## 5. Train

`scale_pos_weight` tells LightGBM to care more about the rare positives, so it
doesn't take the lazy always-say-no route.

**Evaluate with AUPRC, not AUROC.** At 1:20 imbalance AUROC reads ~0.95 for a
model that's barely working. AUPRC is measured against the 4.8% baseline.

In [ ]:
PARAMS = dict(
    objective="binary", metric="average_precision",
    learning_rate=0.05, num_leaves=63, min_data_in_leaf=50,
    feature_fraction=0.3, bagging_fraction=0.8, bagging_freq=1,
    lambda_l2=1.0, verbose=-1, num_threads=os.cpu_count(), seed=RNG,
)

def train_eval(X, y, tr, te, params=PARAMS, rounds=800, label=""):
    spw = (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)
    p = dict(params, scale_pos_weight=spw)
    ds_tr = lgb.Dataset(X[tr], y[tr])
    ds_te = lgb.Dataset(X[te], y[te], reference=ds_tr)
    m = lgb.train(p, ds_tr, num_boost_round=rounds, valid_sets=[ds_te],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    pr = m.predict(X[te], num_iteration=m.best_iteration)
    ap, auc = average_precision_score(y[te], pr), roc_auc_score(y[te], pr)
    base = y[te].mean()
    if label:
        print(f"{label:<28} AUPRC {ap:.4f}  (baseline {base:.4f}, lift {ap/base:5.1f}x)   AUROC {auc:.4f}")
    return m, pr, ap, auc

model, pred, ap, auc = train_eval(X, y, tr_idx, te_idx, label="FULL (window + enzyme)")

## 6. Ablations — the part that decides if this is real

Three deliberate handicaps:

**Enzyme-only** (window blanked). Should be near baseline. If it scores well,
your negatives leak — certain enzymes are mostly-negative and the model reads
enzyme identity alone. That was the failure mode of the original
serine-vs-aspartic negative plan.

**Window-only** (enzyme blanked). Will score moderately — some positions are
intrinsically cleavable. Tells you how much of performance is generic rather
than enzyme-specific.

**Shuffled labels**. Must collapse to baseline. If it doesn't, your splitting
is broken.

In [ ]:
res = {"full": ap}

import gc
X_e = X.copy(); X_e[:, :WIN_FEATS] = 0
_, _, res["enzyme_only"], _ = train_eval(X_e, y, tr_idx, te_idx, rounds=300, label="enzyme only (window=0)")
del X_e; gc.collect()

X_w = X.copy(); X_w[:, WIN_FEATS:] = 0
_, _, res["window_only"], _ = train_eval(X_w, y, tr_idx, te_idx, rounds=300, label="window only (enzyme=0)")
del X_w; gc.collect()

y_sh = y.copy(); y_sh[tr_idx] = np.random.permutation(y_sh[tr_idx])
_, _, res["shuffled"], _ = train_eval(X, y_sh, tr_idx, te_idx, rounds=200, label="shuffled labels (control)")

b = pairs.label.iloc[te_idx].mean()
print("\n--- verdict ---")
print("enzyme-only near baseline :", "PASS" if res["enzyme_only"] < 3*b else "FAIL - negatives leak enzyme identity")
print("shuffled collapses        :", "PASS" if res["shuffled"] < 2*b else "FAIL - split is broken")
print("full beats window-only    :", "PASS" if res["full"] > 1.15*res["window_only"] else "WEAK - enzyme adds little")

## 7. Do the extra negative tiers help?

Tier 1 is in-protein decoys. Tiers 2 and 3 are cross-enzyme hard negatives.
Train on T1 only and compare — if T2/T3 don't improve leave-one-enzyme-out,
drop them.

In [ ]:
t1 = pairs.tier.isin(["positive", "T1_in_protein"]).values
tr1 = np.array([i for i in tr_idx if t1[i]]); te1 = np.array([i for i in te_idx if t1[i]])
_, _, ap_t1, _ = train_eval(X, y, tr1, te1, rounds=500, label="T1 negatives only")
print(f"\nall tiers {ap:.4f} vs T1-only {ap_t1:.4f} (different test sets, compare lift not raw)")

## 8. Leave-one-enzyme-out

Train on 28 enzymes, test on the 29th, never seen. This is whether you learned
pocket chemistry or a 29-entry lookup table.

**Expect this to be much worse than section 5.** That is not failure — with 29
enzymes it's the expected truth, and it's the number to report. Enzymes with a
close relative in training (S08.071 <-> S08.073, both Arg-specific) should do
better than isolated ones.

In [ ]:
rows = []
for e in enz_codes:
    te_m = (pairs.merops_id == e).values
    tr_m = ~te_m
    if te_m.sum() < 30 or pairs.label[te_m].sum() < 5: continue
    _, _, a, _ = train_eval(X, y, np.where(tr_m)[0], np.where(te_m)[0], rounds=300)
    rows.append(dict(enzyme=e, family=pairs.family[te_m].iloc[0], n=int(te_m.sum()),
                     pos=int(pairs.label[te_m].sum()), baseline=pairs.label[te_m].mean(),
                     auprc=a, lift=a/pairs.label[te_m].mean()))

loeo = pd.DataFrame(rows).sort_values("lift", ascending=False)
print(loeo.round(3).to_string(index=False))
print(f"\nmedian LOEO lift: {loeo.lift.median():.2f}x   (1.0 = no generalization)")
print(f"by family:\n{loeo.groupby('family').lift.median().round(2).to_string()}")

## 9. Which features matter

If the top features cluster at window positions 3 and 4 — that's P1 and P1' —
the model found real protease chemistry. If they're spread evenly, be suspicious.

In [ ]:
imp = model.feature_importance("gain")
n_per = WIN_FEATS // Wn
pos_gain = [imp[i*n_per:(i+1)*n_per].sum() for i in range(Wn)]
names = ["P4","P3","P2","P1","P1'","P2'","P3'","P4'"]
tot = sum(pos_gain) + imp[WIN_FEATS:].sum()
for n, g in zip(names, pos_gain):
    print(f"{n:<4} {g/tot:6.1%} {'#'*int(60*g/tot)}")
print(f"{'ENZ':<4} {imp[WIN_FEATS:].sum()/tot:6.1%} {'#'*int(60*imp[WIN_FEATS:].sum()/tot)}")

## 10. Calibration

You asked for probabilities. Because training was weighted toward positives, raw
outputs aren't probabilities — a "0.8" might mean 30% in reality. Isotonic
regression fits the correction on held-out data.

In [ ]:
# proper three-way split by cluster: train / calibrate / holdout
from sklearn.model_selection import GroupShuffleSplit

cl = pairs.cluster.values
uniq = pd.unique(cl)
rs = np.random.RandomState(RNG); rs.shuffle(uniq)
n = len(uniq)
cl_tr  = set(uniq[: int(.70*n)])
cl_cal = set(uniq[int(.70*n): int(.85*n)])
cl_ho  = set(uniq[int(.85*n):])

i_tr  = np.where(pd.Series(cl).isin(cl_tr ).values)[0]
i_cal = np.where(pd.Series(cl).isin(cl_cal).values)[0]
i_ho  = np.where(pd.Series(cl).isin(cl_ho ).values)[0]
assert not (cl_tr & cl_cal) and not (cl_tr & cl_ho) and not (cl_cal & cl_ho)
print(f"train {len(i_tr):,} | calib {len(i_cal):,} | holdout {len(i_ho):,}  (disjoint clusters)")

m_cal, _, _, _ = train_eval(X, y, i_tr, i_cal, label="model for calibration")

# fit the correction on calib, measure honesty on holdout
iso = IsotonicRegression(out_of_bounds="clip").fit(m_cal.predict(X[i_cal]), y[i_cal])
raw = m_cal.predict(X[i_ho])
cal = iso.predict(raw)

print(f"\nholdout AUPRC raw {average_precision_score(y[i_ho], raw):.4f} "
      f"| calibrated {average_precision_score(y[i_ho], cal):.4f}  (ranking is unchanged by design)")

rel = pd.DataFrame({"pred": cal, "true": y[i_ho]})
rel["bin"] = pd.qcut(rel.pred, 10, duplicates="drop")
tbl = rel.groupby("bin", observed=True).agg(predicted=("pred","mean"),
                                            actual=("true","mean"), n=("true","size"))
tbl["gap"] = (tbl.predicted - tbl.actual).abs()
print("\nreliability on untouched holdout:")
print(tbl.round(4).to_string())
print(f"\nmean calibration error: {tbl.gap.mean():.4f}  (want < 0.02)")
print("If predicted and actual diverge badly, the calibration clusters were too few "
      "or too different from training - reshuffle and retry.")

## 11. Save

In [ ]:
OUT = SRC if USE_DRIVE else "."
model.save_model(f"{OUT}/lgbm_cleavage.txt")
import pickle
pickle.dump({"iso": iso, "enz_codes": enz_codes, "meta": meta,
             "pca": ipca, "pca_dim": PCA_DIM, "params": PARAMS}, open(f"{OUT}/postproc.pkl","wb"))
json.dump({"ablations": {k: float(v) for k,v in res.items()},
           "loeo_median_lift": float(loeo.lift.median()),
           "split_method": method, "baseline": float(BASELINE)},
          open(f"{OUT}/results.json","w"), indent=2)
print("saved:", os.listdir(OUT))

## How to read your results

| Signal | Means |
|---|---|
| Full AUPRC > 5x baseline | real, usable signal |
| Enzyme-only near baseline | negatives are clean — the key check |
| Shuffled near baseline | splitting is honest |
| Full >> window-only | enzyme identity is contributing |
| LOEO lift > 2x | some genuine cross-enzyme transfer |
| LOEO lift ~1x | it's a per-enzyme lookup table — still useful, but say so |
| P1/P1' dominate importance | learned protease chemistry, not an artifact |

If enzyme-only or shuffled fail, stop and fix the data. No architecture change
will help.

If everything passes and LOEO is weak, that's the expected state with 29
enzymes. The next step that would actually move it is more enzyme diversity,
not a bigger model.